# Урок 16 - Разгръщане на мащабируеми агенти с Microsoft Foundry

В този бележник ще създадете **готов за продукция агент за обслужване на клиенти** за измислената компания **Contoso**. За разлика от по-ранните уроци, тук не става въпрос за цикъла на разсъждения на агента — става въпрос за всичко *около* него, което прави агента безопасен за работа в голям мащаб:

1. **Извикване на инструменти** — проверка на поръчки и създаване на билети.
2. **RAG** — отговори по политики от база знания.
3. **Памет** — запомняне на клиента през различни ходове.
4. **Рутиране на модели** — изпращане на прости запитвания към малък модел, а сложните към голям модел.
5. **Кеширане на отговори** — обслужване на повторни въпроси без извикване на модел.
6. **Човешко одобрение** — възстановявания над определен праг се спират за подписване.
7. **Евакуационна врата** — офлайн тестов набор, който блокира лошо пускане в експлоатация.
8. **Наблюдателност** — OpenTelemetry проследяване около всяка заявка.

Всяка секция е самостоятелна и изпълнима. Прочетете всеки ред — примитивите за продукция са умишлено малки.


## Настройка

Преди да стартирате тази тетрадка, уверете се, че имате:

1. **Проект в Microsoft Foundry** с внедрен чат модел (напр. `gpt-5-mini`).
2. **Влезли сте чрез Azure CLI** — стартирайте `az login` в терминала си.
3. **Настроени са необходимите променливи на средата:**
   - `AZURE_AI_PROJECT_ENDPOINT` — крайна точка на вашия проект в Microsoft Foundry.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — името на вашия внедрен модел.

Разделът RAG използва **Azure AI Search**, когато са зададени `AZURE_SEARCH_SERVICE_ENDPOINT` и `AZURE_SEARCH_API_KEY`, и преминава към търсене в паметта, за да може тетрадката да работи без Search ресурс.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import re
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential(),
)
print("Foundry client ready.")

## 1. Инструменти

Производствените инструменти вършат реална работа с реални системи. Тук симулираме база данни за поръчки и билетна система с обикновени Python функции. Декораторът `@tool` ги прави достъпни за агента.

Обърнете внимание, че `issue_refund` използва `approval_mode="always_require"` за възстановявания над прага — това е примитивът с човешко участие, който внедряваме по-късно.


In [ ]:
# Simulated backend systems (in production these are API calls behind scoped identities).
ORDERS = {
    "A1001": {"status": "shipped", "total": 42.00, "eta": "2 days"},
    "A1002": {"status": "processing", "total": 128.50, "eta": "5 days"},
    "A1003": {"status": "delivered", "total": 19.99, "eta": "delivered"},
}
TICKETS: list[dict] = []
REFUND_APPROVAL_THRESHOLD = 50.0


@tool(approval_mode="never_require")
def get_order_status(order_id: Annotated[str, "The customer's order ID, e.g. A1001"]) -> str:
    """Look up the status of a customer order."""
    order = ORDERS.get(order_id.upper())
    if not order:
        return f"No order found with ID {order_id}."
    return (
        f"Order {order_id.upper()}: status={order['status']}, "
        f"total=${order['total']:.2f}, eta={order['eta']}."
    )


@tool(approval_mode="never_require")
def open_ticket(
    subject: Annotated[str, "Short subject line for the support ticket"],
    details: Annotated[str, "Full description of the customer's issue"],
) -> str:
    """Open a support ticket for issues that need human follow-up."""
    ticket_id = f"T{1000 + len(TICKETS) + 1}"
    TICKETS.append({"id": ticket_id, "subject": subject, "details": details})
    return f"Ticket {ticket_id} opened: {subject}"


def refund_needs_approval(amount: float) -> bool:
    """Refunds above the threshold require a human approver."""
    return amount > REFUND_APPROVAL_THRESHOLD


@tool(approval_mode="always_require")
def issue_refund(
    order_id: Annotated[str, "The order to refund"],
    amount: Annotated[float, "Refund amount in USD"],
) -> str:
    """Issue a refund. Execution pauses for human approval before it runs."""
    return f"Refund of ${amount:.2f} issued for order {order_id.upper()}."


print("Tools defined.")

## 2. RAG — База знания за политики

Въпроси, свързани с политиката („какъв е вашият срок за връщане?“), трябва да се отговарят от авторитетен източник, а не от паметта на модела. Опаковаме малка база знания като инструмент за търсене.

В продукция това е **Azure AI Search**; тук предоставяме търсене по ключови думи в паметта, за да може тетрадката да работи навсякъде, и автоматично превключваме към Azure AI Search, когато са налични променливите на околната среда.


In [ ]:
KNOWLEDGE_BASE = {
    "returns": "Contoso accepts returns within 30 days of delivery for a full refund. Items must be unused and in original packaging.",
    "shipping": "Standard shipping takes 3-5 business days. Express shipping (1-2 days) is available at checkout for an extra fee.",
    "warranty": "All Contoso electronics carry a 12-month limited warranty covering manufacturing defects.",
    "refund_policy": "Refunds are processed to the original payment method within 5 business days of approval. Refunds over $50 require a supervisor's approval.",
}


def _in_memory_search(query: str) -> str:
    q = query.lower()
    hits = [text for key, text in KNOWLEDGE_BASE.items() if key.replace("_", " ") in q or key in q]
    if not hits:
        # crude keyword fallback so the tool still returns something useful
        hits = [text for text in KNOWLEDGE_BASE.values() if any(w in text.lower() for w in q.split())]
    return "\n".join(hits) if hits else "No matching policy found."


def _azure_search(query: str) -> str:
    from azure.core.credentials import AzureKeyCredential
    from azure.search.documents import SearchClient

    client = SearchClient(
        endpoint=os.environ["AZURE_SEARCH_SERVICE_ENDPOINT"],
        index_name=os.getenv("AZURE_SEARCH_INDEX_NAME", "contoso-policies"),
        credential=AzureKeyCredential(os.environ["AZURE_SEARCH_API_KEY"]),
    )
    results = client.search(search_text=query, top=3)
    return "\n".join(r.get("content", "") for r in results) or "No matching policy found."


USE_AZURE_SEARCH = bool(os.getenv("AZURE_SEARCH_SERVICE_ENDPOINT") and os.getenv("AZURE_SEARCH_API_KEY"))


@tool(approval_mode="never_require")
def search_policies(query: Annotated[str, "The policy question to look up"]) -> str:
    """Search Contoso support policies to answer customer questions."""
    if USE_AZURE_SEARCH:
        return _azure_search(query)
    return _in_memory_search(query)


print(f"RAG ready. Using {'Azure AI Search' if USE_AZURE_SEARCH else 'in-memory search'}.")

## 3. Памет

Агент за поддръжка, който забравя с кого говори, е лош агент за поддръжка. Ние поддържаме малко хранилище с профили за всеки клиент и вмъкваме кратко обобщение в инструкциите на агента. В продукция това е услуга за памет (вж. Урок 13); тук речник прави модела видим.


In [ ]:
CUSTOMER_MEMORY: dict[str, dict] = {
    "cust-42": {"name": "Dana", "tier": "enterprise", "recent_order": "A1002"},
    "cust-99": {"name": "Sam", "tier": "standard", "recent_order": "A1003"},
}


def memory_context(customer_id: str) -> str:
    profile = CUSTOMER_MEMORY.get(customer_id)
    if not profile:
        return "This is a new customer with no history."
    return (
        f"Customer {profile['name']} ({profile['tier']} tier). "
        f"Most recent order: {profile['recent_order']}."
    )


print(memory_context("cust-42"))

## 4 & 5. Насочване на модел и кеширане на отговори

Два ценови лоста, свързани в един обработчик на заявки:

- **Насочване**: евтин евристичен класификатор решава дали една заявка се нуждае от малък или голям модел.
- **Кеширане**: нормализирани повтарящи се въпроси се обслужват директно от кеш без повикване на модел.

Класификаторът тук е умишлено прост. В продукция бихте го валидирали срещу трафик и бихте могли да го замените с Model Router на Foundry.


In [ ]:
SMALL_MODEL = os.getenv("AZURE_AI_SMALL_MODEL", model)   # e.g. gpt-5-nano
LARGE_MODEL = os.getenv("AZURE_AI_LARGE_MODEL", model)   # e.g. gpt-5-mini

response_cache: dict[str, str] = {}
route_counters = {"small": 0, "large": 0, "cache": 0}


def normalize(query: str) -> str:
    return re.sub(r"\s+", " ", query.lower().strip())


COMPLEX_SIGNALS = ("refund", "cancel", "complaint", "escalate", "broken", "wrong", "why")


def is_simple(query: str) -> bool:
    """Route complex or high-stakes requests to the large model; everything else to the small one."""
    q = query.lower()
    if any(signal in q for signal in COMPLEX_SIGNALS):
        return False
    return len(q.split()) <= 20


def choose_model(query: str) -> str:
    return SMALL_MODEL if is_simple(query) else LARGE_MODEL


print(f"Small model: {SMALL_MODEL} | Large model: {LARGE_MODEL}")

## 6 & 8. Агентът, Одобрението от човека и Наблюдаемостта

Сега сглобяваме агента от гореспоменатите инструменти и обграждаме всяка заявка в span на OpenTelemetry. Функцията `handle_support_request` е обработчикът за производствени заявки: кеш → маршрутизиране → проследяване → изпълнение → кеш.

Одобрението от човека се обработва от рамката: тъй като `issue_refund` има `approval_mode="always_require"`, изпълнението се паузира и показва заявка за одобрение, която решаваме изрично.


In [ ]:
# Tracing: use the Agent Framework tracer if available, else a no-op so the notebook runs anywhere.
try:
    from agent_framework.observability import get_tracer
    tracer = get_tracer()
except Exception:  # observability extras not installed
    from contextlib import contextmanager

    class _NoopSpan:
        def set_attribute(self, *_args, **_kwargs):
            pass

    class _NoopTracer:
        @contextmanager
        def start_as_current_span(self, _name):
            yield _NoopSpan()

    tracer = _NoopTracer()


SUPPORT_INSTRUCTIONS = (
    "You are Contoso's customer support agent. Be concise, friendly, and accurate. "
    "Use search_policies for policy questions, get_order_status for orders, "
    "open_ticket when a human needs to follow up, and issue_refund for refunds. "
    "Never invent policy details."
)

# Build one agent per model tier so we can route by cost. The current agent-framework
# selects the model on the client, so each tier gets its own FoundryChatClient.
_TOOLS = [get_order_status, open_ticket, search_policies, issue_refund]
_agents_by_model: dict[str, object] = {}


def agent_for(model_name: str):
    if model_name not in _agents_by_model:
        client = FoundryChatClient(
            project_endpoint=endpoint,
            model=model_name,
            credential=AzureCliCredential(),
        )
        _agents_by_model[model_name] = client.as_agent(
            name="ContosoSupportAgent",
            instructions=SUPPORT_INSTRUCTIONS,
            tools=_TOOLS,
        )
    return _agents_by_model[model_name]


# Default agent (used by the evaluation gate, which does not route).
support_agent = agent_for(SMALL_MODEL)


async def handle_support_request(query: str, customer_id: str) -> str:
    # 1. Serve from cache when we can.
    key = normalize(query)
    if key in response_cache:
        route_counters["cache"] += 1
        return response_cache[key]

    # 2. Route by complexity to control cost.
    chosen_model = choose_model(query)
    route_counters["small" if chosen_model == SMALL_MODEL else "large"] += 1

    # 3. Add per-customer memory to the prompt.
    context = memory_context(customer_id)
    prompt = f"[Customer context: {context}]\n\n{query}"

    # 4. Run inside a trace span for observability.
    with tracer.start_as_current_span("support_request") as span:
        span.set_attribute("customer.id", customer_id)
        span.set_attribute("routed.model", chosen_model)
        response = await agent_for(chosen_model).run(prompt)

    text = response.text
    response_cache[key] = text
    return text


print("Support agent assembled.")

In [ ]:
# Try a few requests. The first is simple (small model), the second is a refund (large model + approval path).
print(await handle_support_request("What is your return window?", "cust-99"))
print("---")
print(await handle_support_request("Where is my order A1002?", "cust-42"))
print("---")
# Repeat the first question -> served from cache.
print(await handle_support_request("What is your return window?", "cust-99"))
print("---")
print("Routing counters:", route_counters)

## 7. Оценъчен шлюз

Това е шлюзът за пускане от урока: офлайн тестов набор оценява агента и пускането в експлоатация продължава само ако процентът на преминаване надвиши прага. Тук оценителят е проста проверка за съвпадение на ключови думи, за да се запази тетрадката самостоятелна; в продукция бихте използвали LLM като съдия или рамкова оценителна система (виж Урок 10).


In [ ]:
TEST_CASES = [
    {"input": "How long do I have to return an item?", "expected": ["30 days", "refund"]},
    {"input": "How fast is standard shipping?", "expected": ["3-5", "business days"]},
    {"input": "What is the status of order A1001?", "expected": ["shipped", "A1001"]},
    {"input": "Do your electronics have a warranty?", "expected": ["12-month", "warranty"]},
]


def score_response(actual: str, expected_keywords: list[str]) -> float:
    actual_l = actual.lower()
    hits = sum(1 for kw in expected_keywords if kw.lower() in actual_l)
    return hits / len(expected_keywords)


async def evaluation_gate(test_cases: list[dict], threshold: float = 0.8) -> bool:
    passed = 0
    for case in test_cases:
        result = await support_agent.run(case["input"])
        s = score_response(result.text, case["expected"])
        status = "PASS" if s >= 0.5 else "FAIL"
        print(f"[{status}] {case['input']}  (score={s:.0%})")
        if s >= 0.5:
            passed += 1
    pass_rate = passed / len(test_cases)
    print(f"\nEvaluation pass rate: {pass_rate:.0%} (gate: {threshold:.0%})")
    return pass_rate >= threshold


gate_passed = await evaluation_gate(TEST_CASES, threshold=0.8)
print("\nDeploy allowed:" , gate_passed)

## Събиране на всичко: Симулирано пускане

Клетката по-долу показва целия цикъл, описан в урока: изпълнете вратата за оценка и "публикувайте" само ако тя премине. Това е моделът, който бихте изпълнили в CI, преди да публикувате версия на агент към Foundry Agent Service.


In [ ]:
async def release(test_cases: list[dict]) -> None:
    print("Running pre-deployment evaluation gate...\n")
    if await evaluation_gate(test_cases, threshold=0.8):
        print("\n✅ Gate passed — promoting agent version to the Foundry Agent Service.")
    else:
        print("\n❌ Gate failed — release blocked. Fix the agent and re-run.")


await release(TEST_CASES)

## Резюме

Съставихте готов за продукция агент за поддръжка на клиенти с интегрирани всички оперативни аспекти:

- **Инструменти, RAG и памет** дават възможности и контекст на агента.
- **Маршрутизиране и кеширане на модели** държат латентността и разходите под контрол.
- **Човешко одобрение** пази високорискови действия като големи възстановявания на суми.
- **Оценъчната врата** блокира лоши версии преди да бъдат пуснати.
- **Проследяване** прави всяка заявка наблюдаема.

### Предизвикателство

Разширете този агент да:

1. **Поддържа множество модели** — добавете трети "разсъдъчен" слой и маршрутизирайте ескалации/реклами към него.
2. **Добавете оценъчни врати** — разширете `TEST_CASES`, за да включва сценарии за одобрение на възстановявания и потвърдете, че вратата улавя регресии.
3. **Добавете маршрутизиране, осъзнаващо разходите** — следете приблизителна цена на заявка (малка срещу голяма срещу кеш) и отпечатайте отчет за разходите след партида смесени заявки.

В следващия урок предприемате обратния път и пускате агент изцяло на собствената си машина с Microsoft Foundry Local и Qwen.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Отказ от отговорност**:
Този документ е преведен с помощта на AI преводачески услуга [Co-op Translator](https://github.com/Azure/co-op-translator). Въпреки че се стремим към точност, моля имайте предвид, че автоматизираните преводи могат да съдържат грешки или неточности. Оригиналният документ на неговия роден език трябва да се счита за авторитетен източник. За критична информация се препоръчва професионален човешки превод. Ние не носим отговорност за каквито и да е недоразумения или неправилни тълкувания, произтичащи от използването на този превод.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
